# 📈 Forecasting com Prophet

---

**Ficha Técnica do Modelo**

| Campo | Valor |
|-------|-------|
| **Modelo** | Prophet — Decomposição aditiva/multiplicativa com sazonalidade automática |
| **Biblioteca** | `prophet` 1.3.0 — `Prophet` |
| **Hiperparâmetros configurados** | `yearly_seasonality=True`, `weekly_seasonality=False`, `daily_seasonality=False`, `seasonality_mode='multiplicative'` |
| **Busca de hiperparâmetros** | Não |
| **Critério de seleção** | N/A |
| **Séries utilizadas** | 29 séries com total ≥ 36 observações (`len(series_data) < TEST_SIZE + 12`) |
| **Horizonte** | 3 meses (`HORIZON = 3`) |
| **Protocolo de avaliação** | Walk-forward expansível, 24 meses de teste (`TEST_SIZE = 24`), janelas de 3 meses |
| **Reprodutibilidade** | `SEED = 42` (`random.seed(42)` + `np.random.seed(42)`) |
| **Referência** | Taylor, S.J. & Letham, B. (2018). Forecasting at Scale. *The American Statistician*, 72(1), 37–45. Meta Open Source. |

---

In [ ]:
# ── Semente global para reprodutibilidade ──
import random, numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"🔒 Seed fixada: {SEED}")

## 1. Instalação de Dependências

In [1]:
# Instalação do Prophet (se necessário)
!pip install prophet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Importação de Bibliotecas

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from tqdm import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("📈 FORECASTING COM PROPHET")
print("="*70)
print(f"📅 Data de execução: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("✅ Bibliotecas carregadas!")

c:\Users\phill\Projetos\lag-llama\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


📈 FORECASTING COM PROPHET
📅 Data de execução: 2026-04-05 13:45:05
✅ Bibliotecas carregadas!


## 3. Carregamento dos Dados

In [3]:
# Carregar base econômica
df = pd.read_csv('base_economica_brasil.csv', index_col=0, parse_dates=True)

# Lista de todas as séries
ALL_SERIES = list(df.columns)
# Excluir PIM e IPCA_mensal da análise
ALL_SERIES = [s for s in ALL_SERIES if s not in ['PIM', 'IPCA_mensal']]

print(f"📊 Base carregada: {len(df)} observações")
print(f"📈 Séries disponíveis: {len(ALL_SERIES)}")
print(f"📅 Período: {df.index.min().strftime('%Y-%m')} a {df.index.max().strftime('%Y-%m')}")
df.head()

📊 Base carregada: 108 observações
📈 Séries disponíveis: 35
📅 Período: 2017-01 a 2025-12


,IBC_Br,Selic,Cambio_USDBRL,Desemprego,Brent_USD,Soja_USD,Minerio_USD,Ibovespa,ICC_FGV,Credito_Total,...,PMC_Ampliado,IGPM,INPC,M2,Divida_PIB,Vendas_Varejo,Balanca_Comercial,NUCI_FGV,EAI_Emprego_Ind,SP500
2017-01-01,90.56860,13.17,3.1270,12.7,55.25,379.589979,80.818182,64671.0,102.25,1537976.0,...,-0.1,0.64,0.42,5.842420e+09,46.46,89.14,2427.0,73.2,514650.2,2278.870117
2017-02-01,90.92437,12.82,3.0993,13.3,53.36,380.872624,88.950000,66662.0,113.80,1535492.0,...,-4.8,0.08,0.24,5.861693e+09,47.26,82.01,4368.2,73.7,1024989.8,2363.639893
2017-03-01,99.65199,12.15,3.1684,13.9,52.20,366.095056,87.195652,64984.0,109.38,1540450.0,...,-1.9,0.01,0.32,5.936526e+09,47.53,88.52,6418.6,73.3,1585673.4,2362.719971
2017-04-01,93.78125,11.59,3.1984,13.7,49.46,347.861310,70.400000,65403.0,109.01,1530470.0,...,-0.5,-1.10,0.08,5.925396e+09,47.48,88.31,6125.7,73.4,2123135.2,2384.199951
2017-05-01,95.21290,11.15,3.2437,13.4,49.40,350.179987,61.630435,62711.0,103.49,1526937.0,...,4.9,-0.93,0.36,5.947256e+09,48.01,90.43,6712.1,74.0,2673694.8,2411.800049


## 4. Configuração do Experimento

In [4]:
# Parâmetros de previsão
HORIZON = 3          # 3 meses à frente
TEST_SIZE = 24       # últimos 24 meses para teste

print(f"⚙️ Configuração:")
print(f"   - Horizonte de previsão: {HORIZON} meses")
print(f"   - Tamanho do teste: {TEST_SIZE} meses")

⚙️ Configuração:
   - Horizonte de previsão: 3 meses
   - Tamanho do teste: 12 meses


In [5]:
def calcular_metricas(real, previsto):
    """Calcula MAE, RMSE e MAPE."""
    real = np.array(real)
    previsto = np.array(previsto)
    
    mae = np.mean(np.abs(real - previsto))
    rmse = np.sqrt(np.mean((real - previsto) ** 2))
    mape = np.mean(np.abs((real - previsto) / (real + 1e-8))) * 100
    
    return mae, rmse, mape

def forecast_with_prophet(train_series, test_series, horizon=HORIZON):
    """
    Realiza previsão usando Prophet com walk-forward validation.
    Retorna previsões para o período de teste.
    """
    all_preds = []
    all_actuals = []
    
    # Preparar dados de treino inicial
    train_data = train_series.copy()
    
    # Walk-forward no período de teste
    for i in range(0, len(test_series), horizon):
        # Preparar dados para Prophet
        prophet_df = pd.DataFrame({
            'ds': train_data.index,
            'y': train_data.values
        })
        
        # Criar e treinar modelo (silencioso)
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode='multiplicative'
        )
        model.fit(prophet_df)
        
        # Criar dataframe futuro
        future = model.make_future_dataframe(periods=horizon, freq='MS')
        forecast = model.predict(future)
        
        # Pegar previsões
        preds = forecast['yhat'].iloc[-horizon:].values
        
        # Valores reais do período
        actual_end = min(i + horizon, len(test_series))
        actuals = test_series.iloc[i:actual_end].values
        
        # Ajustar tamanho das previsões
        preds = preds[:len(actuals)]
        
        all_preds.extend(preds)
        all_actuals.extend(actuals)
        
        # Expandir treino com dados reais para próxima iteração
        train_data = pd.concat([train_data, test_series.iloc[i:actual_end]])
    
    return np.array(all_preds), np.array(all_actuals)

print("✅ Funções definidas!")

✅ Funções definidas!


## 5. Treinamento e Previsão (Walk-Forward)

In [6]:
# Dicionário para armazenar resultados
all_results = {}

print("="*80)
print("🚀 INICIANDO PREVISÕES COM PROPHET")
print("="*80)

for series_name in tqdm(ALL_SERIES, desc="Processando séries"):
    try:
        series_data = df[series_name].dropna()
        
        if len(series_data) < TEST_SIZE + 12:
            print(f"⚠️ {series_name}: Poucos dados ({len(series_data)} obs)")
            continue
        
        # Dividir em treino e teste
        train_series = series_data.iloc[:-TEST_SIZE]
        test_series = series_data.iloc[-TEST_SIZE:]
        
        # Realizar previsão
        preds, actuals = forecast_with_prophet(train_series, test_series, horizon=HORIZON)
        
        if len(preds) > 0:
            # Calcular métricas
            mae, rmse, mape = calcular_metricas(actuals, preds)
            
            all_results[series_name] = {
                'mae': mae,
                'rmse': rmse,
                'mape': mape,
                'n_points': len(preds),
                'preds': preds,
                'actuals': actuals,
                'test_index': test_series.index[:len(preds)]
            }
            print(f"✅ {series_name}: MAE={mae:.2f}, RMSE={rmse:.2f}, MAPE={mape:.2f}%")
        else:
            print(f"❌ {series_name}: Sem resultados válidos")
            
    except Exception as e:
        print(f"❌ {series_name}: Erro - {str(e)[:50]}")

print("\n" + "="*80)
print(f"✅ CONCLUÍDO: {len(all_results)}/{len(ALL_SERIES)} séries processadas")
print("="*80)

🚀 INICIANDO PREVISÕES COM PROPHET


Processando séries:   0%|          | 0/35 [00:00<?, ?it/s]13:45:07 - cmdstanpy - INFO - Chain [1] start processing
13:45:11 - cmdstanpy - INFO - Chain [1] done processing
13:45:13 - cmdstanpy - INFO - Chain [1] start processing
13:45:16 - cmdstanpy - INFO - Chain [1] done processing
13:45:17 - cmdstanpy - INFO - Chain [1] start processing
13:45:18 - cmdstanpy - INFO - Chain [1] done processing
13:45:20 - cmdstanpy - INFO - Chain [1] start processing
13:45:20 - cmdstanpy - INFO - Chain [1] done processing
Processando séries:   3%|▎         | 1/35 [00:15<08:47, 15.51s/it]

✅ IBC_Br: MAE=1.75, RMSE=2.18, MAPE=1.60%


13:45:22 - cmdstanpy - INFO - Chain [1] start processing
13:45:24 - cmdstanpy - INFO - Chain [1] done processing
13:45:25 - cmdstanpy - INFO - Chain [1] start processing
13:45:27 - cmdstanpy - INFO - Chain [1] done processing
13:45:28 - cmdstanpy - INFO - Chain [1] start processing
13:45:29 - cmdstanpy - INFO - Chain [1] done processing
13:45:31 - cmdstanpy - INFO - Chain [1] start processing
13:45:31 - cmdstanpy - INFO - Chain [1] done processing
Processando séries:   6%|▌         | 2/35 [00:26<07:12, 13.12s/it]

✅ Selic: MAE=2.72, RMSE=2.98, MAPE=19.30%


...

✅ NUCI_FGV: MAE=2.66, RMSE=2.81, MAPE=3.25%


13:52:47 - cmdstanpy - INFO - Chain [1] start processing
13:52:50 - cmdstanpy - INFO - Chain [1] done processing
13:52:52 - cmdstanpy - INFO - Chain [1] start processing
13:52:55 - cmdstanpy - INFO - Chain [1] done processing
13:52:56 - cmdstanpy - INFO - Chain [1] start processing
13:52:56 - cmdstanpy - INFO - Chain [1] done processing
13:52:58 - cmdstanpy - INFO - Chain [1] start processing
13:52:59 - cmdstanpy - INFO - Chain [1] done processing
Processando séries:  97%|█████████▋| 34/35 [07:53<00:12, 12.18s/it]

✅ EAI_Emprego_Ind: MAE=39060.49, RMSE=46356.86, MAPE=0.63%


13:53:00 - cmdstanpy - INFO - Chain [1] start processing
13:53:03 - cmdstanpy - INFO - Chain [1] done processing
13:53:05 - cmdstanpy - INFO - Chain [1] start processing
13:53:09 - cmdstanpy - INFO - Chain [1] done processing
13:53:10 - cmdstanpy - INFO - Chain [1] start processing
13:53:11 - cmdstanpy - INFO - Chain [1] done processing
13:53:12 - cmdstanpy - INFO - Chain [1] start processing
13:53:12 - cmdstanpy - INFO - Chain [1] done processing
Processando séries: 100%|██████████| 35/35 [08:07<00:00, 13.93s/it]

✅ SP500: MAE=312.68, RMSE=367.46, MAPE=4.85%

✅ CONCLUÍDO: 35/35 séries processadas


## 6. Resultados e Métricas

In [7]:
# Criar DataFrame com resultados
results_df = pd.DataFrame([
    {
        'Série': name,
        'MAE': data['mae'],
        'RMSE': data['rmse'],
        'MAPE (%)': data['mape'],
        'N Pontos': data['n_points']
    }
    for name, data in all_results.items()
]).sort_values('MAPE (%)')

# Adicionar classificação
def classificar(mape):
    if mape < 10:
        return '⭐ Excelente'
    elif mape < 20:
        return '✅ Muito Bom'
    elif mape < 30:
        return '👍 Bom'
    elif mape < 50:
        return '⚠️ Regular'
    else:
        return '❌ Difícil'

results_df['Classificação'] = results_df['MAPE (%)'].apply(classificar)
results_df = results_df.set_index('Série')

# Exibir tabela
print("="*80)
print("📊 RANKING - PROPHET")
print("="*80)
print(f"\nModelo: Prophet | Horizonte: {HORIZON} meses | Teste: {TEST_SIZE} meses\n")
print(results_df.round(2).to_string())

# Estatísticas gerais
print("\n" + "-"*80)
print("📈 ESTATÍSTICAS GERAIS:")
print(f"   Total de séries analisadas: {len(results_df)}")
print(f"   MAE médio geral: {results_df['MAE'].mean():.2f}")
print(f"   RMSE médio geral: {results_df['RMSE'].mean():.2f}")
print(f"   MAPE médio geral: {results_df['MAPE (%)'].mean():.2f}%")
print(f"   Melhor série (MAPE): {results_df['MAPE (%)'].idxmin()} ({results_df['MAPE (%)'].min():.2f}%)")
print(f"   Pior série (MAPE): {results_df['MAPE (%)'].idxmax()} ({results_df['MAPE (%)'].max():.2f}%)")

📊 RANKING - PROPHET

Modelo: Prophet | Horizonte: 3 meses | Teste: 12 meses

                             MAE          RMSE      MAPE (%)  N Pontos Classificação
Série                                                                               
CPI_USA             4.400000e-01  4.900000e-01  1.400000e-01        12   ⭐ Excelente
EAI_Emprego_Ind     3.906049e+04  4.635686e+04  6.300000e-01        12   ⭐ Excelente
Massa_Salarial      2.527330e+03  2.932360e+03  7.100000e-01        12   ⭐ Excelente
Prod_Ind_USA        8.900000e-01  1.160000e+00  8.800000e-01        12   ⭐ Excelente
M2                  1.277190e+08  1.445848e+08  9.200000e-01        12   ⭐ Excelente
Credito_Total       3.969453e+04  4.375892e+04  1.020000e+00        12   ⭐ Excelente
PMS_Volume          1.250000e+00  1.490000e+00  1.140000e+00        12   ⭐ Excelente
Divida_PIB          7.800000e-01  9.100000e-01  1.240000e+00        12   ⭐ Excelente
IBC_Br              1.750000e+00  2.180000e+00  1.600000e+00        12   

## 7. Visualização: Ranking MAPE por Série

In [ ]:
# ── Gráfico: Ranking MAPE por Série ──
sorted_df = results_df.sort_values('MAPE (%)')

fig, ax = plt.subplots(figsize=(12, 8))

cores = ['#2ecc71' if m < 10 else '#3498db' if m < 20 else '#f39c12' if m < 30 else '#e74c3c'
         for m in sorted_df['MAPE (%)']]

bars = ax.barh(range(len(sorted_df)), sorted_df['MAPE (%)'],
               color=cores, edgecolor='white', height=0.7)
ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels(sorted_df.index)
ax.invert_yaxis()
ax.set_xlabel('MAPE (%)')
ax.set_title(f'Prophet — MAPE por Série\n(Walk-Forward, h={HORIZON}, teste={TEST_SIZE} meses)',
             fontsize=13, fontweight='bold')
ax.axvline(x=sorted_df['MAPE (%)'].mean(), color='red', linestyle='--',
           label=f'Média: {sorted_df["MAPE (%)"].mean():.1f}%')
ax.legend(loc='lower right')

for i, (bar, val) in enumerate(zip(bars, sorted_df['MAPE (%)'])):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('prophet_mape_por_serie.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: prophet_mape_por_serie.png")

## 8. Visualização: Real vs. Projetado (Top 6 Séries)

In [ ]:
# ── Gráfico: Real vs. Projetado (Top 6 Séries por MAPE) ──
top6 = sorted_df.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, sn in zip(axes.flatten(), top6):
    data = all_results[sn]

    # Valores reais (teste)
    ax.plot(range(len(data['actuals'])), data['actuals'], 'b-o',
            label='Real', markersize=4, linewidth=2)

    # Previsões do modelo
    ax.plot(range(len(data['actuals'])), data['preds'], 'r--s',
            label='Previsão', markersize=4, linewidth=2)

    ax.set_title(f"{sn}\nMAPE: {data['mape']:.1f}%", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45, labelsize=8)

axes.flatten()[0].legend(fontsize=8)
fig.suptitle('Prophet — Real vs. Projetado (6 Melhores Séries)',
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('prophet_previsoes.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: prophet_previsoes.png")

## 9. Exportação de Resultados

In [10]:
# Salvar resultados com nomes padronizados para compatibilidade com consolidação
results_export = results_df.reset_index()
results_export.columns = ['Serie', 'MAE', 'RMSE', 'MAPE', 'N_Pontos', 'Classificacao']
results_export.to_csv('resultados_prophet.csv', index=False)
print("💾 Resultados salvos em 'resultados_prophet.csv'")
print(f"   Colunas: {list(results_export.columns)}")

# Salvar previsões individuais (Serie, Data, Previsao) para análises complementares
previsoes_rows = []
for serie, data in all_results.items():
    for d, p in zip(data['test_index'], data['preds']):
        previsoes_rows.append({'Serie': serie, 'Data': str(d)[:10], 'Previsao': p})
df_prev = pd.DataFrame(previsoes_rows)
df_prev.to_csv('previsoes_prophet.csv', index=False)
print(f"💾 Previsões salvas em 'previsoes_prophet.csv' ({len(df_prev)} linhas)")

💾 Resultados salvos em 'resultados_prophet.csv'
   Colunas: ['Serie', 'MAE', 'RMSE', 'MAPE', 'N_Pontos', 'Classificacao']
💾 Previsões salvas em 'previsoes_prophet.csv' (420 linhas)
